In [104]:
import numpy as np
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV
from sklearn.datasets import load_iris, fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')
import pickle
from utils import get_classifier_metrics
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV


## Paso 1: Carga del conjunto de datos

In [3]:
df = pd.read_csv('../data/raw/playstore_reviews.csv')
df.head()

,package_name,review,polarity
0,com.facebook.katana,privacy at least put some option appear offli...,0
1,com.facebook.katana,"messenger issues ever since the last update, ...",0
2,com.facebook.katana,profile any time my wife or anybody has more ...,0
3,com.facebook.katana,the new features suck for those of us who don...,0
4,com.facebook.katana,forced reload on uploading pic on replying co...,0


- `package_name` Nombre de la aplicación móvil (categórico)
- `review` Comentario sobre la aplicación móvil (categórico)
- `polarity` Variable de clase (0 o 1), siendo 0 un comentario negativo y 1, positivo (categórico numérico)

## Paso 2: Estudio de variables y su contenido

En este caso, tenemos solo 3 variables: 2 predictoras y una etiqueta dicotómica. De las dos predictoras, realmente solo nos interesa la parte del comentario, ya que el hecho de clasificar un comentario en positivo o negativo dependerá de su contenido, no de la aplicación de la que se haya escrito. Por lo tanto, la variable package_name habría que eliminarla.

Cuando trabajamos con textos como en este caso, no tiene sentido hacer un EDA, el proceso es diferente, ya que la única variable que nos interesa es la que contiene el texto. En otros casos en los que el texto formase parte de un conjunto complejo con otras variables predictoras numéricas y el objetivo de predicción sea distinto, entonces tiene sentido aplicar un EDA.

Sin embargo, no podemos trabajar con texto plano, antes hay que procesarlo. Este proceso consta de varios pasos:

In [4]:
df.drop(['package_name'], axis=1, inplace=True)
df

,review,polarity
0,privacy at least put some option appear offli...,0
1,"messenger issues ever since the last update, ...",0
2,profile any time my wife or anybody has more ...,0
3,the new features suck for those of us who don...,0
4,forced reload on uploading pic on replying co...,0
...,...,...
886,loved it i loooooooooooooovvved it because it...,1
887,all time legendary game the birthday party le...,1
888,ads are way to heavy listen to the bad review...,0
889,fun works perfectly well. ads aren't as annoy...,1


#### 1. Eliminar espacios y convertir a minúsculas el texto:

In [ ]:
df['review'] = df['review'].str.strip().str.lower()

,review,polarity
0,privacy at least put some option appear offlin...,0
1,"messenger issues ever since the last update, i...",0
2,profile any time my wife or anybody has more t...,0
3,the new features suck for those of us who don'...,0
4,forced reload on uploading pic on replying com...,0
...,...,...
886,loved it i loooooooooooooovvved it because it ...,1
887,all time legendary game the birthday party lev...,1
888,ads are way to heavy listen to the bad reviews...,0
889,fun works perfectly well. ads aren't as annoyi...,1


#### 2. Dividir el conjunto de datos en train y test: X_train, X_test, y_train, y_test

In [21]:
X = df['review']
y = df['polarity']

In [22]:
# Split
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=18)

In [23]:
X_train.shape

(712,)

In [24]:
X_test.shape

(179,)

#### 3. Transformar el texto en una matriz de recuento de palabras. Esta es una forma de obtener características numéricas a partir del texto. Para ello, utilizamos el conjunto de train para entrenar el transformador y la aplicamos en test:

In [25]:
vectorizer_model = CountVectorizer(stop_words = "english")

In [26]:
X_train = vectorizer_model.fit_transform(X_train).toarray()

In [27]:
X_test = vectorizer_model.transform(X_test).toarray()

## Paso 3: Construye un naive bayes

Implementar el modelo elegido (GaussianNB, MultinomialNB o BernoulliNB)

In [28]:
model_gaussian = GaussianNB()

In [29]:
model_gaussian.fit(X_train, y_train)

,priors,None
,var_smoothing,1e-09


In [44]:
y_pred_train_gaussian = model_gaussian.predict(X_train)
y_pred_train_gaussian

array([0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0,
       1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1,
       0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1,
       1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0,
       1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0,
       0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0,
       0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1,
       0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1,
       1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1,
       1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0,
       1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1,
       0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1,

In [45]:
y_pred_test_gaussian = model_gaussian.predict(X_test)
y_pred_test_gaussian

array([1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1,
       0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0,
       0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1,
       0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0,
       0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0,
       0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1,
       1, 0, 0])

In [ ]:
metrics_gaussian = {'Accuracy Train': accuracy_score(y_train, y_pred_train_gaussian),
                    'Accuracy Test': accuracy_score(y_test, y_pred_test_gaussian)}

metrics_gaussian

{'Accuracy Train': 0.9845505617977528, 'Accuracy Test': 0.7877094972067039}

Aplicar otros dos modelos (MultinomialNB o BernoulliNB)

In [35]:
model_multinomial = MultinomialNB()

In [36]:
model_multinomial.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [47]:
y_pred_train_multinomial = model_multinomial.predict(X_train)

In [48]:
y_pred_test_multinomial = model_multinomial.predict(X_test)

In [ ]:
metrics_multinomial = {'Accuracy Train': accuracy_score(y_train, y_pred_train_multinomial),
                       'Accuracy Test': accuracy_score(y_test, y_pred_test_multinomial)}

metrics_: 

{'Accuracy Train': 0.9550561797752809, 'Accuracy Test': 0.8156424581005587}

In [43]:
model_bernoulli = BernoulliNB()

In [50]:
model_bernoulli.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,binarize,0.0
,fit_prior,True
,class_prior,None


In [53]:
y_pred_train_bernoulli = model_bernoulli.predict(X_train)
y_pred_train_bernoulli

array([0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0,
       0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0,
       1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0,
       0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1,
       0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1,
       0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,

In [54]:
y_pred_test_bernoulli = model_bernoulli.predict(X_test)
y_pred_test_bernoulli

array([0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1,
       0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
       0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0])

In [58]:
metrics_bernoulli = {'Accuracy Train': accuracy_score(y_train, y_pred_train_bernoulli),
                     'Accuracy Test': accuracy_score(y_test, y_pred_test_bernoulli)}

metrics_bernoulli

{'Accuracy Train': 0.9073033707865169, 'Accuracy Test': 0.7150837988826816}

> #### Observaciones:
>
> - Se probaron las tres variantes de Naive Bayes: GaussianNB, MultinomialNB y BernoulliNB.
> - GaussianNB asume que los datos siguen una distribución normal, lo que no es adecuado para datos de texto, donde las características son conteos de palabras.
> - BernoulliNB funciona bien con variables binarias (presencia/ausencia de palabras), pero nuestro dataset usa frecuencias de palabras, por lo que no aprovecha completamente la información.
> - MultinomialNB es la implementación más adecuada para datos de texto con conteo de palabras. En las pruebas realizadas, MultinomialNB obtuvo la mejor precisión y métricas de evaluación más altas, confirmando que es la opción óptima para este dataset.  
 
**Conclusión** Para problemas de clasificación de texto con frecuencia de palabras, la elección de MultinomialNB está justificada tanto teóricamente como empíricamente.

## Paso 4: Optimiza el modelo anterior (el mejor - MultinomialNB)

In [70]:
hyperparams = {"alpha": np.linspace(0.01, 10.0, 200),
               "fit_prior": [True, False]}

# We initialize the random search
random_search = RandomizedSearchCV(model_multinomial, 
                                   hyperparams,
                                   n_iter=50,
                                   scoring="accuracy",
                                   cv=5,
                                   random_state=18)
random_search

,estimator,MultinomialNB()
,param_distributions,"{'alpha': array([ 0.01 ... 10. ]), 'fit_prior': [True, False]}"
,n_iter,50
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,18
,error_score,nan


In [73]:
random_search.fit(X_train, y_train)

,estimator,MultinomialNB()
,param_distributions,"{'alpha': array([ 0.01 ... 10. ]), 'fit_prior': [True, False]}"
,n_iter,50
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,18
,error_score,nan


In [74]:
random_search.best_params_

{'fit_prior': True, 'alpha': np.float64(0.1606030150753769)}

In [75]:
best_model = random_search.best_estimator_
best_model

,alpha,np.float64(0.1606030150753769)
,force_alpha,True
,fit_prior,True
,class_prior,None


In [76]:
# We train the best model
random_search.best_estimator_.fit(X_train, y_train)

,alpha,np.float64(0.1606030150753769)
,force_alpha,True
,fit_prior,True
,class_prior,None


In [77]:
y_pred_test_random = random_search.best_estimator_.predict(X_test)
y_pred_test_random

array([1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1,
       0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1,
       0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0,
       0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0,
       0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1,
       1, 0, 0])

In [78]:
y_pred_train_random = random_search.best_estimator_.predict(X_train)
y_pred_train_random

array([0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0,
       1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1,
       0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1,
       1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0,
       1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0,
       0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1,
       0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1,
       1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1,
       1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0,
       0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1,
       0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1,

In [81]:
get_classifier_metrics(y_pred_test_random, y_test, y_pred_train_random, y_train)

,Accuracy,F1 Score,Precision,Recall
Train set,0.978933,0.978933,0.978933,0.978933
Test set,0.815642,0.815642,0.815642,0.815642


> #### Observaciones sobre RandomizedSearchCV en MultinomialNB:
>
> - Antes de aplicar RandomizedSearchCV, el modelo tenía un accuracy de entrenamiento de 0.955, mientras que después del ajuste de hiperparámetros aumentó a 0.979.
> - El accuracy en test se mantuvo igual (0.816) después del Random Search. Aunque el modelo se ajustó mejor al entrenamiento, no hubo sobreajuste significativo.


## Paso 5: Guarda el modelo

In [82]:
with open('/workspaces/inetke-machine-learning/models/naive-bayes-playstore-reviews.pkl', 'wb') as file:
    pickle.dump(best_model, file)

## Paso 6: Explora otras alternativas

Otro modelo que podríamos aplicar sería Logistic Regression, ya que es muy popular para clasificación de texto.

In [98]:
y_train.dtypes

dtype('int64')

In [99]:
model_logistic_regression = LogisticRegression(random_state=18)

In [100]:
model_logistic_regression.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,18
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [101]:
y_train_pred_logistic = model_logistic_regression.predict(X_train)
y_test_pred_logistic = model_logistic_regression.predict(X_test)

In [112]:
model_accuracy = accuracy_score(y_test, y_test_pred_logistic)
model_accuracy

0.7932960893854749

In [102]:
get_classifier_metrics(y_test_pred_logistic, y_test, y_train_pred_logistic, y_train)

,Accuracy,F1 Score,Precision,Recall
Train set,1.000000,1.000000,1.000000,1.000000
Test set,0.793296,0.793296,0.793296,0.793296


In [ ]:
# We define the parameters we want to adjust manually
hyperparams_logistic = {'C': [0.001, 0.01, 0.1, 1, 10, 100, 1000],
               'penalty': ['l1', 'l2', 'elasticnet', None],
               'solver': ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']}

# We initialise the grid
grid = GridSearchCV(model_logistic_regression,
                    hyperparams_logistic,
                    scoring="accuracy",
                    cv=5)
grid

,estimator,LogisticRegre...ndom_state=18)
,param_grid,"{'C': [0.001, 0.01, ...], 'penalty': ['l1', 'l2', ...], 'solver': ['newton-cg', 'lbfgs', ...]}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


In [119]:
grid.fit(X_train, y_train)

grid.best_params_

{'C': 0.001, 'penalty': None, 'solver': 'sag'}

In [120]:
grid.best_estimator_

,penalty,None
,dual,False
,tol,0.0001
,C,0.001
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,18
,solver,'sag'
,max_iter,100
,multi_class,'deprecated'


In [121]:
best_model_grid = grid.best_estimator_

y_pred_train_grid = best_model_grid.predict(X_train)
y_pred_test_grid = best_model_grid.predict(X_test)

grid_accuracy = accuracy_score(y_train, y_pred_train_grid)

grid_accuracy

0.9943820224719101

In [122]:
get_classifier_metrics(y_pred_test_grid, y_test, y_pred_train_grid, y_train)

,Accuracy,F1 Score,Precision,Recall
Train set,0.994382,0.994382,0.994382,0.994382
Test set,0.804469,0.804469,0.804469,0.804469


> #### Conclusión: 
>
> - MultinomialNB generaliza mejor, Logistic Regression alcanza casi 1.0 en train, pero baja más en test, existe sobreajuste. 
> - MultinomialNB sigue siendo más robusto para datos de conteo de palabras. 
